# Mini-TP 7 — Seguridad con SAIF: Data Lake + gRPC (Sesión 7)

La consigna pide asegurar **una** capa. Aseguro **dos**, las que más expone mi servicio:

- **Parte A: Data Lake (MinIO/S3).** Desde el Mini-TP 6 el servicio **carga el
  modelo desde el lake**: si alguien lee o reemplaza el artefacto en `models/`,
  compromete todo lo que sirven REST, GraphQL, gRPC y el streaming.
- **Parte B: gRPC.** El servidor del Mini-TP 3 escucha con `add_insecure_port`:
  sin autenticación y en texto plano.

| | Data Lake | gRPC |
|---|---|---|
| **Riesgo** | exfiltración del modelo, acceso excesivo, artefacto manipulado | acceso no autenticado, *man-in-the-middle*, abuso con mensajes enormes |
| **Controles** | SHA-256 verificado al cargar · URL prefirmada con expiración · usuarios por rol con acceso mínimo por prefijo · audit log | JWT en la metadata verificado por un interceptor, con permisos por rol · mTLS · límite de tamaño de mensaje · audit log por RPC |
| **Elementos SAIF** | bases sólidas · coherencia de controles | bases sólidas · coherencia de controles · detectar y actuar |

Código: `src/model_service/lake_security.py` (parte A) y `src/model_service/grpc_security.py` (parte B).

**Requisito:** MinIO en `localhost:9000` (usuario `minio` / `minio123`), como en el Mini-TP 6.

In [1]:
import os, requests
os.environ.setdefault("GRPC_VERBOSITY", "ERROR")   # silencia los logs de handshake de gRPC
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

from model_service import datalake as dl
from model_service import lake_security as ls

root = dl.client()
dl.create_zones(root)
print("MinIO:", requests.get(f"{dl.ENDPOINT}/minio/health/live", timeout=3).status_code, "| bucket:", dl.BUCKET)

MinIO: 200 | bucket: mlops2-lake


# Parte A — Data Lake

## 1. Versión vulnerable: el bucket público

Lo más fácil para "compartir el modelo" es abrir el prefijo `models/` con una
política anónima. Cualquiera que conozca la URL se lleva el artefacto, sin
credenciales y sin dejar rastro.

In [2]:
import json

public_policy = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow", "Principal": {"AWS": ["*"]}, "Action": ["s3:GetObject"],
    "Resource": [f"arn:aws:s3:::{dl.BUCKET}/models/*"]}]}
root.put_bucket_policy(Bucket=dl.BUCKET, Policy=json.dumps(public_policy))

direct_url = f"{dl.ENDPOINT}/{dl.BUCKET}/models/v1/model.joblib"
r = requests.get(direct_url)
print(f"GET anónimo -> HTTP {r.status_code}, {len(r.content)} bytes  <- el modelo completo, robado")

GET anónimo -> HTTP 200, 337064 bytes  <- el modelo completo, robado


In [3]:
root.delete_bucket_policy(Bucket=dl.BUCKET)
print(f"Sin política pública -> HTTP {requests.get(direct_url).status_code}")

Sin política pública -> HTTP 403


## 2. Integridad: publicar con checksum y verificar al cargar

`model.joblib` es un pickle, y **deserializar un pickle ejecuta código**.
Si alguien con acceso de escritura reemplaza el artefacto, el servicio
ejecutaría lo que esa persona quiera al arrancar. Por eso el servicio compara
el SHA-256 publicado **antes** de llamar a `joblib.load`.

In [4]:
admin = ls.AuditedLake("admin", root)
digest = ls.publish_model(admin, "v1")
print("publicado models/v1 con sha256", digest[:16], "…")

art = ls.load_verified_model(admin, "v1")
print("cargado y verificado:", art["metadata"]["model_name"], art["metadata"]["model_version"])

publicado models/v1 con sha256 d57502c7aff5bbfa …
cargado y verificado: iris-rf 1.0.0


Ataque: alguien sube un artefacto alterado a la misma key (sin tocar el `.sha256`).

In [5]:
original = admin.get("models/v1/model.joblib")
admin.put("models/v1/model.joblib", original[:-16] + b"\x00" * 16)   # artefacto manipulado

try:
    ls.load_verified_model(admin, "v1")
except ls.IntegrityCheckError as e:
    print("BLOQUEADO antes de deserializar ->", e)

admin.put("models/v1/model.joblib", original)   # restauro el artefacto bueno
print("restaurado:", ls.load_verified_model(admin, "v1")["metadata"]["model_version"])

BLOQUEADO antes de deserializar -> models/v1: sha256 de9d78f786bc… != d57502c7aff5…
restaurado: 1.0.0


> El checksum detecta la alteración **accidental o de alguien que no puede
> escribir el `.sha256`**. Un atacante con escritura sobre todo `models/` podría
> recalcularlo. Por eso se combina con la sección 4: el serving no puede escribir
> en `models/`, y en producción el hash se fijaría fuera del lake (en la metadata
> de MLflow o firmado).

## 3. Control principal (guiado): URL prefirmada con expiración

En vez de abrir el bucket se entrega un enlace **firmado, para una sola key y
que caduca**. Sin la firma, MinIO responde 403.

In [6]:
import time

url = ls.presigned_url(admin, "models/v1/model.joblib", expires_in=3)
print("URL prefirmada:", url[:80], "…")
print("dentro del plazo  -> HTTP", requests.get(url).status_code)
time.sleep(4)
print("tras expirar      -> HTTP", requests.get(url).status_code)
print("sin firma         -> HTTP", requests.get(url.split("?")[0]).status_code)

URL prefirmada: http://localhost:9000/mlops2-lake/models/v1/model.joblib?X-Amz-Algorithm=AWS4-HM …
dentro del plazo  -> HTTP 200
tras expirar      -> HTTP 403
sin firma         -> HTTP 403


## 4. Mi entrega (Data Lake)

### TODO 3: zonas con permisos distintos (mínimo privilegio)

No me quedé con la política "conceptual": creo en MinIO **un usuario por rol**
con una política IAM limitada a sus prefijos. Cada etapa del pipeline sólo
puede tocar su zona:

| Rol | Lee | Escribe |
|---|---|---|
| `ingestor` | — | `raw/` |
| `etl` | `raw/` | `curated/` |
| `training` | `curated/` | `models/` |
| `serving` | `curated/`, `models/` | `audit/serving/` |

In [7]:
pols = ls.create_roles()
print(json.dumps(pols["serving"], indent=2))

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject"
      ],
      "Resource": [
        "arn:aws:s3:::mlops2-lake/curated/*",
        "arn:aws:s3:::mlops2-lake/models/*"
      ]
    },
    {
      "Effect": "Allow",
      "Action": [
        "s3:PutObject"
      ],
      "Resource": [
        "arn:aws:s3:::mlops2-lake/audit/serving/*"
      ]
    }
  ]
}


In [8]:
def try_access(role, op, key):
    s3 = ls.role_client(role)
    try:
        if op == "get":
            s3.get_object(Bucket=dl.BUCKET, Key=key)
        else:
            s3.put_object(Bucket=dl.BUCKET, Key=key, Body=b"intento")
        result = "permitido"
    except Exception as e:
        result = "DENEGADO" if "AccessDenied" in str(e) else type(e).__name__
    print(f"{role:<14} {op:<4} {key:<28} -> {result}")

try_access("serving",       "get", "models/v1/model.joblib")
try_access("serving",       "put", "models/v1/model.joblib")   # no puede reemplazar el modelo
try_access("serving",       "get", "raw/iris/iris.csv")        # no ve el dato crudo
try_access("ingestor",      "put", "raw/eventos/e1.json")
try_access("ingestor",      "get", "models/v1/model.joblib")   # no puede exfiltrar el modelo
try_access("training",      "get", "curated/iris/iris.csv")
try_access("training",      "get", "raw/iris/iris.csv")
_ = root.delete_object(Bucket=dl.BUCKET, Key="raw/eventos/e1.json")  # limpio la prueba

serving        get  models/v1/model.joblib       -> permitido
serving        put  models/v1/model.joblib       -> DENEGADO
serving        get  raw/iris/iris.csv            -> DENEGADO
ingestor       put  raw/eventos/e1.json          -> permitido
ingestor       get  models/v1/model.joblib       -> DENEGADO
training       get  curated/iris/iris.csv        -> permitido
training       get  raw/iris/iris.csv            -> DENEGADO


### TODO 1: mi modelo en el lake con checksum, cargado por el serving

El serving usa **sus propias credenciales** (no las de root) y carga el
`iris-rf` verificado, como lo haría la API al arrancar.

In [9]:
serving = ls.AuditedLake("serving", ls.role_client("serving"))
art = ls.load_verified_model(serving, "v1")
pipe, meta = art["pipeline"], art["metadata"]

x = [[5.1, 3.5, 1.4, 0.2], [6.7, 3.0, 5.2, 2.3]]
print("predicción:", [meta["target_names"][p] for p in pipe.predict(x)])

try:                                   # el serving intenta pisar el modelo
    serving.put("models/v1/model.joblib", b"payload")
except Exception as e:
    print("serving -> models/:", "AccessDenied" if "AccessDenied" in str(e) else e)

predicción: ['setosa', 'virginica']
serving -> models/: AccessDenied


### TODO 2: registro de acceso (audit log)

`AuditedLake` registra cada `get`/`put`/`verify`/`presign` con timestamp,
actor, key y resultado. Los **intentos denegados también quedan registrados**,
que es lo que más interesa en una auditoría. El serving persiste su registro
en `audit/serving/`, la única zona donde puede escribir.

In [10]:
import pandas as pd

key = serving.persist()
print("persistido en", f"s3://{dl.BUCKET}/{key}\n")
pd.DataFrame(admin.records + serving.records)[["ts", "actor", "op", "key", "ok", "detail"]]

persistido en s3://mlops2-lake/audit/serving/20261009T002739.jsonl



,ts,actor,op,key,ok,detail
0,2026-10-09T00:27:33.818+00:00,admin,put,models/v1/model.joblib,True,337064 bytes
1,2026-10-09T00:27:33.828+00:00,admin,put,models/v1/model.sha256,True,64 bytes
2,2026-10-09T00:27:33.833+00:00,admin,get,models/v1/model.joblib,True,337064 bytes
3,2026-10-09T00:27:33.836+00:00,admin,get,models/v1/model.sha256,True,64 bytes
4,2026-10-09T00:27:33.839+00:00,admin,verify,models/v1/model.joblib,True,d57502c7aff5
5,2026-10-09T00:27:35.110+00:00,admin,get,models/v1/model.joblib,True,337064 bytes
6,2026-10-09T00:27:35.122+00:00,admin,put,models/v1/model.joblib,True,337064 bytes
7,2026-10-09T00:27:35.124+00:00,admin,get,models/v1/model.joblib,True,337064 bytes
8,2026-10-09T00:27:35.126+00:00,admin,get,models/v1/model.sha256,True,64 bytes
9,2026-10-09T00:27:35.127+00:00,admin,verify,models/v1/model.joblib,False,checksum distinto


# Parte B — gRPC

## 5. Versión vulnerable: el servidor del Mini-TP 3

Levanto el mismo `grpc_server.serve()` de siempre (`add_insecure_port`, sin
auth) y pongo en el medio un **proxy espía**: un proxy TCP que reenvía el
tráfico y guarda una copia, como alguien en la misma red. Cualquier cliente
puntúa sin credenciales, y el espía lee la respuesta **y el token** en texto
plano.

In [11]:
import grpc, tempfile
from model_service import grpc_security as gs, scoring_pb2 as pb, scoring_pb2_grpc as rpc
from model_service.grpc_server import serve

f = pb.Features(sepal_length_cm=6.7, sepal_width_cm=3.0, petal_length_cm=5.2, petal_width_cm=2.3)
app_token = gs.issue_token("app-web", "predictor")       # JWT legítimo de la app

server_port, spy_port = gs.free_port(), gs.free_port()
insecure_server = serve(server_port)
spy = gs.SpyProxy(spy_port, server_port)          # cliente -> espía -> servidor

stub = rpc.ScoringStub(grpc.insecure_channel(f"localhost:{spy_port}"))
print("sin credenciales  ->", stub.Predict(f, timeout=3).prediction_label)
stub.Predict(f, metadata=gs.with_token(app_token), timeout=3)

wire = bytes(spy.captured)
print(f"el espía capturó {len(wire)} bytes")
print("  ¿ve la predicción 'virginica'?", b"virginica" in wire)
print("  ¿ve el JWT completo?          ", app_token.encode() in wire)
insecure_server.stop(0); spy.close()

sin credenciales  -> virginica
el espía capturó 1127 bytes
  ¿ve la predicción 'virginica'? True
  ¿ve el JWT completo?           True


Además el servidor acepta mensajes de hasta 4 MB por defecto: un cliente
puede mandar lotes enormes a `PredictBatch` y acaparar CPU (DoS), o hacer
miles de consultas para copiar el modelo (*model extraction*).

## 6. Control principal: interceptor con JWT + límite de tamaño

La versión guiada compara contra un token fijo (`"Bearer tok-valido"`). La
cambio por un **JWT** (TODO 1), que es como se hace en producción:

- **`issue_token`** firma con HS256 los claims `sub` (quién), `role`, `iss` y
  `exp` (vence a los 5 minutos). El secreto sale de `JWT_SECRET` en el entorno.
- **`AuthInterceptor`** hace `jwt.decode` **antes** de ejecutar la RPC:
  verifica la firma, el vencimiento y el emisor. Si algo falla corta con
  `UNAUTHENTICATED`. Después mira el claim `role`:

  | Rol | Puede llamar |
  |---|---|
  | `predictor` | `Predict` |
  | `batch` | `Predict`, `PredictBatch` |

- **`max_receive_message_length = 64 KB`**: suficiente para un lote de ~1.800
  vectores; lo que supere eso se rechaza antes de llegar al servicer.
- El servicer es el del Mini-TP 3 con **mi modelo** `iris-rf` y el mensaje
  `Features` con sus 4 features (TODO 2).
- **mTLS**: lo agrego en la sección 7 y lo uso desde ya.

In [12]:
import jwt, base64, json

certs = gs.generate_certs(tempfile.mkdtemp(prefix="tp7-certs-"))
secure_port = gs.free_port()
server, audit = gs.serve_secure(secure_port, certs, mtls=True)
stub = rpc.ScoringStub(gs.secure_channel(secure_port, certs))

def call(name, fn):
    try:
        print(f"{name:<36} -> OK:", fn())
    except grpc.RpcError as e:
        print(f"{name:<36} -> {e.code().name}: {e.details()}")

batch_token = gs.issue_token("batch-job", "batch")
APP, BATCH = gs.with_token(app_token), gs.with_token(batch_token)
print("claims del JWT de app-web:", {k: v for k, v in gs.verify_token(app_token).items() if k in ("sub", "role", "iss")})
batch = lambda n: pb.FeatureBatch(items=[f] * n)

claims del JWT de app-web: {'sub': 'app-web', 'role': 'predictor', 'iss': 'model-service'}


Tokens que **no** deberían pasar:

- **Firmado con otro secreto**: alguien arma su propio JWT con `role: batch`.
- **Adulterado**: tomo el JWT legítimo de `app-web` y cambio `role` a `batch`
  en el payload sin poder volver a firmarlo.
- **Vencido**: un JWT legítimo cuyo `exp` ya pasó (por ejemplo, uno robado de un log viejo).

In [13]:
forged = jwt.encode({"sub": "atacante", "role": "batch", "iss": "model-service", "exp": 4102444800},
                    "otro-secreto-que-adivine-0123456789", algorithm="HS256")

header, payload, signature = app_token.split(".")
claims = json.loads(base64.urlsafe_b64decode(payload + "=="))
claims["role"] = "batch"
tampered_payload = base64.urlsafe_b64encode(json.dumps(claims).encode()).rstrip(b"=").decode()
tampered = f"{header}.{tampered_payload}.{signature}"

expired = gs.issue_token("app-web", "predictor", ttl_seconds=-60)

call("Predict sin token", lambda: stub.Predict(f, timeout=3))
call("Predict con JWT de otro secreto", lambda: stub.Predict(f, metadata=gs.with_token(forged), timeout=3))
call("PredictBatch con JWT adulterado", lambda: list(stub.PredictBatch(batch(10), metadata=gs.with_token(tampered), timeout=3)))
call("Predict con JWT vencido", lambda: stub.Predict(f, metadata=gs.with_token(expired), timeout=3))

Predict sin token                    -> UNAUTHENTICATED: falta el header authorization: Bearer <JWT>
Predict con JWT de otro secreto      -> UNAUTHENTICATED: token inválido (InvalidSignatureError)
PredictBatch con JWT adulterado      -> UNAUTHENTICATED: token inválido (InvalidSignatureError)
Predict con JWT vencido              -> UNAUTHENTICATED: token vencido


Tokens legítimos, con el rol que corresponde y con uno que no alcanza, más el límite de tamaño:

In [14]:
call("Predict como app-web (predictor)", lambda: stub.Predict(f, metadata=APP, timeout=3).prediction_label)
call("PredictBatch como app-web (predictor)", lambda: list(stub.PredictBatch(batch(10), metadata=APP, timeout=3)))
call("PredictBatch(100) como batch-job", lambda: len(list(stub.PredictBatch(batch(100), metadata=BATCH, timeout=30))))
print(f"\nlote de 5000 = {batch(5000).ByteSize():,} bytes (límite {gs.MAX_MSG_BYTES:,})")
call("PredictBatch(5000) como batch-job", lambda: len(list(stub.PredictBatch(batch(5000), metadata=BATCH, timeout=30))))

Predict como app-web (predictor)     -> OK: virginica
PredictBatch como app-web (predictor) -> PERMISSION_DENIED: el rol 'predictor' no puede llamar /scoring.Scoring/PredictBatch
PredictBatch(100) como batch-job     -> OK: 100

lote de 5000 = 190,000 bytes (límite 65,536)
PredictBatch(5000) como batch-job    -> RESOURCE_EXHAUSTED: SERVER: Received message larger than max (190000 vs. 65536)


## 7. TLS/mTLS contra el man-in-the-middle

El JWT sólo protege si viaja cifrado: en la sección 5 el espía lo copió
entero del cable y podía reusarlo hasta que venciera (TODO 3, que la consigna
pide conceptual, lo implemento). `generate_certs` crea una **CA propia**, un certificado de servidor para
`localhost` y uno de cliente. Con **mTLS** se autentican las dos puntas: el
cliente verifica al servidor y el servidor exige el certificado del cliente.

Repito el espionaje, ahora contra el servidor seguro:

In [15]:
spy_port = gs.free_port()
spy = gs.SpyProxy(spy_port, secure_port)
spied_stub = rpc.ScoringStub(gs.secure_channel(spy_port, certs))
print("predicción a través del espía:", spied_stub.Predict(f, metadata=APP, timeout=3).prediction_label)

wire = bytes(spy.captured)
print(f"el espía capturó {len(wire)} bytes")
print("  ¿ve la predicción 'virginica'?", b"virginica" in wire)
print("  ¿ve el JWT completo?          ", app_token.encode() in wire)
spy.close()

predicción a través del espía: virginica
el espía capturó 5979 bytes
  ¿ve la predicción 'virginica'? False
  ¿ve el JWT completo?           False


Y los clientes que no cumplen con el canal no llegan ni a presentar el token:

In [16]:
no_cert_stub = rpc.ScoringStub(gs.secure_channel(secure_port, certs, with_client_cert=False))
plaintext_stub = rpc.ScoringStub(grpc.insecure_channel(f"localhost:{secure_port}"))

call("TLS sin certificado de cliente", lambda: no_cert_stub.Predict(f, metadata=APP, timeout=3))
call("canal en texto plano", lambda: plaintext_stub.Predict(f, metadata=APP, timeout=3))

TLS sin certificado de cliente       -> UNAVAILABLE: failed to connect to all addresses; last error: UNAVAILABLE: ipv4:127.0.0.1:36317: Socket closed
canal en texto plano                 -> UNAVAILABLE: failed to connect to all addresses; last error: UNAVAILABLE: ipv4:127.0.0.1:36317: Socket closed


### Audit log por RPC

El interceptor envuelve cada RPC y registra actor, método, código de salida y
latencia **real** (incluye la inferencia). Los rechazos de auth también quedan,
con el motivo (sin token, firma inválida, vencido, rol insuficiente). Es la
medida de **gobernanza** de esta capa.
Dos rechazos no aparecen, y es esperable: el lote de 5000 lo corta el
transporte antes de llegar al interceptor, y las conexiones sin mTLS mueren
en el handshake TLS.

In [17]:
server.stop(0)
pd.DataFrame(audit.records)

,ts,actor,rpc,code,ms,detail
0,2026-10-09T00:27:40.095+00:00,anónimo,Predict,UNAUTHENTICATED,0.00,sin token
1,2026-10-09T00:27:40.097+00:00,anónimo,Predict,UNAUTHENTICATED,0.00,InvalidSignatureError
2,2026-10-09T00:27:40.098+00:00,anónimo,PredictBatch,UNAUTHENTICATED,0.00,InvalidSignatureError
3,2026-10-09T00:27:40.101+00:00,anónimo,Predict,UNAUTHENTICATED,0.00,token vencido
4,2026-10-09T00:27:40.134+00:00,app-web,Predict,OK,27.72,rol predictor
5,2026-10-09T00:27:40.135+00:00,app-web,PredictBatch,PERMISSION_DENIED,0.00,rol predictor
6,2026-10-09T00:27:41.720+00:00,batch-job,PredictBatch,OK,1575.63,rol batch
7,2026-10-09T00:27:41.902+00:00,app-web,Predict,OK,15.41,rol predictor


# Reflexión

## Parte A: Data Lake

**1. ¿Por qué una URL prefirmada con expiración es más segura que un bucket público?**
El bucket público no tiene identidad ni fecha de vencimiento: cualquiera que
adivine o filtre la URL descarga el modelo para siempre, y nadie sabe quién lo
hizo. La URL prefirmada da acceso a **una sola key**, durante **un plazo corto**
y con la firma de una credencial concreta. Si se filtra, el daño se limita a esa
ventana, y generarla queda en el audit log. El bucket sigue privado por defecto.

**2. ¿Qué ataque detecta el checksum del modelo?**
La **manipulación del artefacto** (tampering o supply chain): reemplazar
`model.joblib` por una versión envenenada o por un pickle con código malicioso
que se ejecuta al cargarlo. Como el hash se verifica antes de `joblib.load`, el
artefacto alterado nunca se deserializa. También detecta corrupción accidental
en la subida. No alcanza solo: necesita que quien puede escribir el modelo no
pueda reescribir el hash, y por eso va junto a los permisos por zona.

**3. ¿Qué elemento de SAIF cubre mi control principal?**
**Bases sólidas de seguridad** (*strong security foundations*): llevar al
ecosistema de IA controles clásicos de infraestructura (acceso mínimo,
integridad de artefactos, credenciales temporales). Además la **coherencia de
controles** (*harmonize platform level controls*): la misma política por
prefijo protege el modelo para todos los consumidores (REST, GraphQL, gRPC,
streaming), en lugar de que cada API implemente la suya. El audit log suma
**gobernanza** y la base para **detectar y actuar** sobre accesos anómalos.

## Parte B: gRPC

**1. ¿Qué parte del ataque frena el interceptor y qué parte frena TLS?**
El interceptor frena el **acceso no autorizado**: sin un JWT con firma válida,
vigente y con un rol que alcance, nadie llega al modelo. Un token inventado,
adulterado o vencido se rechaza. Pero no protege el canal: en
la sección 5 el espía leyó el token y la predicción del cable, y con ese token
podía hacerse pasar por `app-web` hasta que el JWT venciera. TLS frena el **man-in-the-middle**: cifra el
tráfico (el espía sólo captura bytes cifrados, sin el JWT ni la predicción) y autentica al
servidor. Con mTLS el servidor también exige certificado de cliente, y el
canal en texto plano ni siquiera conecta. Hacen falta los dos: un token robado
no sirve sin certificado, y un canal cifrado no dice qué puede hacer cada cliente.

**2. ¿Para qué sirve el límite de tamaño de mensaje?**
Corta el **DoS por payload**: un lote gigante a `PredictBatch` se rechaza con
`RESOURCE_EXHAUSTED` en la capa de transporte, antes de deserializarlo y de
gastar CPU en inferencia. También pone un techo a cuánto puede extraer un
cliente en una sola llamada (*model extraction*). En el integrador importa más,
porque las imágenes viajan como `bytes` y el límite se fija según el tamaño
máximo de frame aceptado.

**3. ¿Qué elemento de SAIF cubre mi control principal?**
**Bases sólidas de seguridad**: autenticación, cifrado en tránsito y límites de
recursos, que son controles clásicos de infraestructura aplicados al endpoint
del modelo. Al hacerlo con un interceptor (y no dentro de cada RPC) se gana
**coherencia de controles**: `Predict`, `PredictBatch` y cualquier RPC futura
heredan la misma política. El audit log por RPC aporta **detectar y actuar**:
un actor con muchos `UNAUTHENTICATED` o lotes anormales es una alerta.